# TO-Agents Lite — run it on a free GPU

A multi-agent topology-optimization pipeline: describe a structural problem in prose,
and a group of agents builds the config, runs a 3D optimization, **looks at the rendered
result with a vision model**, proposes revisions, re-runs, and scores the candidates.

The original needs **four A100s**. This notebook needs none of your own hardware:

| Piece | Where it runs |
|---|---|
| TO solver | this Colab runtime — free T4 if you enable one, else CPU |
| Vision agent | Gemini API |
| AI judge | Gemini API |
| Structured output (prose → JSON) | Gemini API |
| 3D rendering | headless Chromium, CPU |

**Before you start** — `Runtime ▸ Change runtime type ▸ T4 GPU`. It works without one,
but the solver drops to a much smaller mesh.

You need **one** free API key:
- Gemini — https://aistudio.google.com/apikey

(Together is optional — only if you'd rather run the JSON step on Llama-3.3-70B.)

### Running it

**Runtime ▸ Run all** works, with one wrinkle: installing pins `numpy<2`, and if Colab
already had numpy 2.x loaded the kernel must restart to pick it up. Run All stops at that
point. So:

1. Add your key to 🔑 **Secrets** as `GEMINI_API_KEY` (see step 6) — otherwise the key
   cell blocks waiting for typed input and Run All stalls there instead.
2. **Runtime ▸ Run all**
3. If the runtime restarts, **Run all again.** The second pass skips the install (it
   detects the packages), skips the restart (numpy is now correct), and runs straight
   through to the app in well under a minute.

Total setup is ~4 minutes.

## 1. What hardware did we get?

In [ ]:
import subprocess

HAS_GPU = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
if HAS_GPU:
    print(subprocess.run(
        'nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv',
        shell=True, capture_output=True, text=True).stdout)
else:
    print('No GPU attached.')
    print('This still works — the solver falls back to pyFANTOM CPU at a smaller mesh.')
    print('For a GPU: Runtime > Change runtime type > T4 GPU, then re-run from here.')

## 2. Clone the repo

Cloned to a **fixed absolute path**, and skipped if it already exists — so re-running this
cell cannot nest clones inside each other.

In [ ]:
import os

REPO_DIR = '/content/to-agents-lite'

if os.path.isdir(REPO_DIR + '/.git'):
    print('already cloned — pulling latest')
    !cd {REPO_DIR} && git pull -q
else:
    !git clone -q https://github.com/bellastewart/to-agents-lite.git {REPO_DIR}

%cd {REPO_DIR}
print('working dir:', os.getcwd())

## 3. Install

One pip resolution pass, then pyFANTOM with `--no-deps`. Both details matter:

- **One pass.** `autogen-agentchat==0.2.40` requires `numpy<2` via flaml. Installing
  piecemeal leaves a scipy built for numpy 2.x and you get
  `module 'numpy' has no attribute 'long'`.
- **`--no-deps` for pyFANTOM.** Its metadata requires `scikit-sparse`, which is
  source-only on PyPI — no wheels for any Python — and compiles against SuiteSparse.
  A missing `cholmod.h` aborts the whole install. `backend.py` stubs the import and
  switches MultiGrid's coarse solver to scipy's `splu`, which is verified working with
  scikit-sparse entirely absent.

Nothing here compiles, so this is just wheel downloads (~2 min).

### Expect red text here — most of it is harmless

pip will complain that Colab's own preinstalled packages want `numpy>=2.0`:

```
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile / tobler / xarray-einstats ... same
```

**Ignore those.** This pipeline does not use jax, tifffile, tobler or xarray-einstats.
`autogen-agentchat 0.2.40` and `numba` both require `numpy<2`, so numpy 1.26 is the
correct outcome, not a mistake.

You will also see `pyfantom requires jupyter / pynvml / scikit-sparse, which is not
installed` — that is `--no-deps` doing its job. pyFANTOM never imports any of the three.

What would be a **real** problem is numpy ending up >= 2.0; the cell checks and repairs
that itself.

In [ ]:
import importlib.util as _u

# Skip the whole install when it is already satisfied — makes a second
# Run All (after a restart) take seconds instead of minutes.
_need = [m for m in ('pyFANTOM','k3d','vtk','playwright','autogen','instructor')
         if _u.find_spec(m) is None]
if _need:
    !pip install -q -r requirements-lite.txt 2>&1 | tail -5
    !pip install -q --no-deps 'pyFANTOM @ git+https://github.com/bellastewart/pyFANTOM_TO-Agents' 2>&1 | tail -3
else:
    print('already installed — skipping')

if HAS_GPU:
    # MUST be <14. cupy-cuda12x 14.x requires numpy>=2.0, but
    # autogen-agentchat 0.2.40 requires numpy<2 — installing 14.x silently
    # upgrades numpy and breaks autogen AND numba. 13.6.0 accepts
    # numpy<2.6,>=1.22, which is the combination that actually works.
    !pip install -q 'cupy-cuda12x<14' 2>&1 | tail -3
    print('installed cupy 13.x for the GPU solver')
else:
    print('skipped cupy — CPU solver')

import importlib.util as _u
missing = [m for m in ('pyFANTOM','k3d','vtk','playwright','autogen','instructor')
           if _u.find_spec(m) is None]
if missing:
    raise SystemExit(f'these failed to install: {missing} — see output above')

# numpy must still be <2: autogen-agentchat 0.2.40 and numba both require it,
# and a stray dependency can quietly pull numpy 2.x back in.
import subprocess, sys
_v = subprocess.run([sys.executable,'-c','import numpy;print(numpy.__version__)'],
                    capture_output=True, text=True).stdout.strip()
print('numpy:', _v)
if _v and int(_v.split('.')[0]) >= 2:
    print('  fixing: something pulled numpy 2.x back in')
    !pip install -q 'numpy<2'

print('\nall required packages present')

## 4. Restart the runtime (only if needed)

The install downgraded numpy to <2. If Colab had already imported numpy 2.x into this
kernel, the old version is still in memory and scipy/numba will misbehave — the classic
symptom is `module 'numpy' has no attribute 'long'`.

The cell below checks whether that is actually the case and restarts **only if it has**.

> **A restart is normal here, not a failure.** Colab announces it as
> *"Your session crashed for an unknown reason"* or *"Runtime was restarted"*. That
> message is expected. Nothing is lost — the repo and the installed packages live on
> disk. **Just carry on with cell 5.**

In [ ]:
import sys, importlib.metadata as _md

_installed = _md.version('numpy')
_loaded = sys.modules.get('numpy')

if _loaded is None:
    print(f'numpy {_installed} not yet imported in this kernel — no restart needed.')
elif _loaded.__version__ == _installed:
    print(f'numpy {_installed} already active — no restart needed.')
else:
    print(f'in memory: numpy {_loaded.__version__}, on disk: {_installed}')
    print('Restarting the kernel so the installed version is used.')
    print('>>> This is EXPECTED. Ignore any "session crashed" message and run cell 5.')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)

## 5. Headless browser

The 3D screenshots the vision agent looks at are rendered by k3d into HTML and captured
with headless Chromium. This is pure CPU — it never needed a GPU.

In [ ]:
# The restart above reset the working directory and cleared HAS_GPU.
%cd /content/to-agents-lite
import os, subprocess
HAS_GPU = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
print('GPU present:', HAS_GPU)

!playwright install-deps chromium > /dev/null 2>&1
!playwright install chromium 2>&1 | tail -2
print('chromium ready')

## 6. Your API keys

**For Run All to work, use Colab Secrets** (the 🔑 icon in the left sidebar):

1. Click 🔑, then **+ Add new secret**
2. Name: `GEMINI_API_KEY`, Value: your key from https://aistudio.google.com/apikey
3. Toggle **Notebook access** on

The cell reads it automatically. If no secret is set it falls back to a hidden prompt —
which works fine when running cells by hand, but stops a Run All in its tracks.

Either way the key never appears in the notebook output, so this file stays safe to
share or commit.

In [ ]:
# @title Pick a tier, then supply its one key  { display-mode: "form" }
# free = one Gemini key, no card, ~6-10 runs/day
# plus = one OpenRouter key, needs a card, ~$0.002/run, no daily cap
# auto = use whichever key you provide (prefers plus)
TIER = "auto"  # @param ["auto", "free", "plus"]

import os, sys
sys.path.insert(0, REPO_DIR)
import tiers


def _secret(name):
    """Read a Colab Secret by exact name. Returns '' if absent."""
    try:
        from google.colab import userdata
        return userdata.get(name) or ''
    except Exception:
        return ''


# Secret names are case-sensitive, so accept the obvious spellings
# ("Openrouter", "OPENROUTER", "OpenRouter", ...) rather than demanding one.
_found = tiers.normalise_keys(lookup=_secret)
for _canonical, _alias in sorted(_found.items()):
    _note = '' if _alias == _canonical else f'  (found as {_alias!r})'
    print(f'{_canonical:20s} set{_note}')
if not _found:
    print('no keys found yet')

# Still nothing for the tier we need? Ask, but only when not on 'auto', so
# Run All never blocks.
_needed = {'free': 'GEMINI_API_KEY', 'plus': 'OPENROUTER_API_KEY'}.get(TIER)
if _needed and not os.environ.get(_needed):
    import getpass
    _v = getpass.getpass(_needed + ' (required for tier ' + TIER + '): ').strip()
    if _v:
        os.environ[_needed] = _v
        print(f'{_needed:20s} set (from prompt)')

_resolved = tiers.detect_tier() if TIER == 'auto' else TIER
if _resolved is None:
    raise SystemExit(
        'No usable key found.\n'
        '  free tier -> GEMINI_API_KEY     https://aistudio.google.com/apikey (no card)\n'
        '  plus tier -> OPENROUTER_API_KEY https://openrouter.ai/keys (needs a card)\n'
        'Add one via the key icon in the left sidebar (any of these spellings\n'
        'works: OPENROUTER_API_KEY, OPENROUTER, Openrouter), then re-run.')
if tiers.missing_keys(_resolved):
    raise SystemExit(f'Tier {_resolved!r} needs: {", ".join(tiers.missing_keys(_resolved))}')

TIER = _resolved
print()
print(tiers.describe_tier(TIER))


## 7. Wire up the models

Role assignment, matching the original design as closely as your credentials allow:

| Role | Where it runs |
|---|---|
| Structured output | Together Llama-3.3-70B (no Gemini quota) |
| Vision | local Qwen2.5-VL on `:8000` if reachable, else Gemini |
| **Judge** | **Gemini** — the role it had originally |

The Gemini free tier allows **20 requests per day per _model_**
(`GenerateRequestsPerDayPerProjectPerModel-FreeTier`), not per account. Three models
means three independent budgets. If one role starts returning 429, change only that
role's model — these are all verified vision-capable with separate quotas:
`gemini-3-flash-preview`, `gemini-3.1-flash-lite`, `gemini-3.5-flash-lite`,
`gemini-3.6-flash`, `gemini-flash-latest`.

Two things worth knowing:

- Together's **serverless** tier has no vision models. They appear in the catalog but need
  a paid dedicated endpoint, so anything image-related must go to Gemini.
- `instructor`'s `JSON_SCHEMA` mode does **not** work against Google's OpenAI-compat
  endpoint (it rejects `response_format.schema`). `TO_INSTRUCTOR_MODE` is auto-set to
  `TOOLS` for Gemini, which is verified working.

To use Together for the JSON step instead, set `TO_TEXT_PROVIDER='together'` and
`TO_TEXT_MODEL='meta-llama/Llama-3.3-70B-Instruct-Turbo'` (needs the optional key above).

In [ ]:
# Apply the tier. Both tiers give the three roles three DIFFERENT models --
# the judge scores what the vision agent described, so shared weights would
# mean shared blind spots.
_applied = tiers.apply_tier(TIER)

# A local vLLM on :8000 is the original vision backend and costs no quota at
# all, so it wins over the tier's hosted choice when it is actually reachable.
import socket as _s
with _s.socket() as _sk:
    _sk.settimeout(1)
    _local_vlm = _sk.connect_ex(('127.0.0.1', 8000)) == 0
if _local_vlm:
    os.environ['TO_VISION_PROVIDER'] = 'vllm'
    os.environ['TO_VISION_MODEL'] = 'Qwen/Qwen2.5-VL-7B-Instruct'
    print('local vLLM found on :8000 - vision uses it instead of the hosted model')

os.environ.update({
    'TO_BACKEND':  'auto',   # CUDA when a usable GPU is present, else CPU
    'TO_WEB_PORT': '8765',   # avoid 8080: Colab tooling and most dev servers use it
    'TO_MAX_RUNS': '3',      # revision rounds; raise once you know the timing
})

print()
for _role in ('text', 'vision', 'judge'):
    _r = _role.upper()
    print(f'{"structured" if _role == "text" else _role:11s} -> '
          f'{os.environ[f"TO_{_r}_PROVIDER"]}/{os.environ[f"TO_{_r}_MODEL"]}')

if TIER == 'free':
    print()
    print('  Free tier: Gemini allows 20 requests/day PER MODEL. Each role uses')
    print('  a different model, so they do not compete - roughly 6-10 full runs')
    print('  per day. If one role starts returning 429, only that role is out.')
else:
    print()
    print('  Plus tier: ~$0.002 per run (~450 runs per $1), no daily cap.')
    print('  Structured output and judge are the SAME models the 4-GPU build')
    print('  used; only vision is substituted. Set a spend limit on your key.')
print('configured')


## 8. Preflight

`doctor.py` independently checks the solver backend, the renderer, and all three model
roles, then reports which tier you qualify for. **If anything is red, stop here** — its
messages name the fix. Everything downstream assumes this passed.

In [ ]:
!python doctor.py

## 9. Launch

Starts the server and opens it in a Colab window. First run also JIT-compiles the numba
kernels, so give it a moment.

In [ ]:
import time, socket, subprocess

PORT = int(os.environ['TO_WEB_PORT'])

server = subprocess.Popen(
    ['python', 'app.py'], env=os.environ.copy(),
    stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

def up(port, timeout=300):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            return False   # died during startup
        with socket.socket() as s:
            s.settimeout(1)
            if s.connect_ex(('127.0.0.1', port)) == 0:
                return True
        time.sleep(2)
    return False

if not up(PORT):
    print('server failed to start — last 40 log lines:')
    print(open('server.log').read()[-4000:])
else:
    print(f'server up on :{PORT}')
    # serve_kernel_port_as_window is deprecated and prints a plain
    # https://localhost:PORT/ link, which does not resolve for you -- the port
    # lives on the Colab VM, not your machine. proxyPort returns the real
    # externally reachable URL, so print that first and always.
    try:
        from google.colab.output import eval_js
        url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
        print()
        print('  OPEN THIS:')
        print(f'  {url}')
    except Exception as e:
        print(f'(could not build a proxy URL: {e})')
        print(f'not on Colab? try http://127.0.0.1:{PORT}/')

    # And embed it inline as well, so there is a working view even if the
    # pop-up is blocked.
    try:
        from google.colab import output
        output.serve_kernel_port_as_iframe(PORT, height=900)
    except Exception:
        pass


## 10. Use it

Pick a built-in example or paste your own description, then press **Run pipeline**.

- **Run** tab — who is speaking and what they are doing, the intended setup diagram,
  the optimization movie, and the final depth/stress renders per revision
- **Activity log** tab — raw stdout
- **Stop** — halts at the current iteration and keeps whatever was produced

### If something goes wrong

```python
print(open('server.log').read()[-5000:])   # server-side traceback
```

### Known limits — read before trusting a result

- **CPU runs are slow and use a reduced mesh** (48×24×24 vs 128×64×64). The agent loop is
  identical; the resolution is not.
- **`LocalFilter` is CUDA-only.** On CPU a per-element `r_min` collapses to its mean and
  prints a warning — length-scale control becomes uniform. It is not the same problem.
- **`MinimumCompliance` drops `E_local`, `local_volume_constraint`, `passive_solid`,
  `passive_void` on CPU**, loudly, because that backend does not accept them.
- **Gemini's free tier has real quotas.** Heavy use returns HTTP 429
  ("exceeded your current quota"). It resets on its own; `doctor.py` reports it
  distinctly from a bad key.
- **Colab recycles idle runtimes.** A long run can be killed mid-way; lower `TO_MAX_RUNS`
  or keep the tab active.

Source, and the measurements behind each of these: https://github.com/bellastewart/to-agents-lite